<a href="https://colab.research.google.com/github/MatiasO2312/AA---Tierra-del-Fuego---Parque-Nacional/blob/main/notebooks/01_exploracion_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploración de Datasets — Parque Nacional Tierra del Fuego

Este notebook realiza la carga, inspección y descripción inicial de los tres
datasets utilizados en el proyecto de Aprendizaje Automático.

**Fuente:** IPIEC — Instituto Provincial de Análisis e Investigación, Estadística y Censos de Tierra del Fuego.

**Datasets:**
- `16_1_04` — Visitas al Parque Nacional TDF por condición de residencia (2015–2026)
- `16_1_01` — Pernoctaciones e ingreso de viajeros en Ushuaia (2004–2026)
- `22_2_01` — Meteorología: temperatura, precipitaciones y días con nieve (2009–2025)

In [1]:
from google.colab import files
uploaded = files.upload()

Saving 22_2_01_Meteorologia_Temperatura_Precipitaciones.xlsx to 22_2_01_Meteorologia_Temperatura_Precipitaciones.xlsx
Saving 16_1_04_Visitas-al-Parque-Nacional-TDF.xlsx to 16_1_04_Visitas-al-Parque-Nacional-TDF.xlsx
Saving 16_1_01_Pernoctaciones_ingresos_estadia_promedio.xlsx to 16_1_01_Pernoctaciones_ingresos_estadia_promedio.xlsx


In [2]:
import pandas as pd

# ── Dataset 1: Visitas al Parque Nacional TDF ────────────────────────────
df_visitas = pd.read_excel('16_1_04_Visitas-al-Parque-Nacional-TDF.xlsx',
                           sheet_name='16_1_04', skiprows=4, header=None)
df_visitas.columns = ['año', 'mes', 'total_visitas', 'drop', 'residentes', 'no_residentes']
df_visitas = df_visitas.drop(columns=['drop'])
df_visitas['año'] = df_visitas['año'].ffill()
df_visitas = df_visitas[df_visitas['mes'].notna() & df_visitas['total_visitas'].notna()]
for col in ['total_visitas', 'residentes', 'no_residentes']:
    df_visitas[col] = pd.to_numeric(df_visitas[col], errors='coerce')
df_visitas['año'] = pd.to_numeric(df_visitas['año'], errors='coerce')
df_visitas = df_visitas[df_visitas['año'] >= 2015]

print("=== VISITAS AL PN TDF ===")
print(f"Filas: {len(df_visitas)} | Columnas: {list(df_visitas.columns)}")
print(f"Período: {int(df_visitas['año'].min())} – {int(df_visitas['año'].max())}")
df_visitas.info()

=== VISITAS AL PN TDF ===
Filas: 135 | Columnas: ['año', 'mes', 'total_visitas', 'residentes', 'no_residentes']
Período: 2015 – 2026
<class 'pandas.core.frame.DataFrame'>
Index: 135 entries, 1 to 135
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   año            135 non-null    int64  
 1   mes            135 non-null    object 
 2   total_visitas  132 non-null    float64
 3   residentes     131 non-null    float64
 4   no_residentes  127 non-null    float64
dtypes: float64(3), int64(1), object(1)
memory usage: 6.3+ KB


In [6]:
# ── Dataset 2: Meteorología ────────────────────────────────────
df_meteo = pd.read_excel('22_2_01_Meteorologia_Temperatura_Precipitaciones.xlsx',
                         sheet_name='por mes', skiprows=8, header=None)

df_meteo.columns = ['año', 'mes',
                    'rg_temp_max', 'rg_temp_min', 'rg_temp_media', '_1', '_2',
                    'rg_lluvia_mm', 'rg_dias_nieve',
                    '_3',
                    'ush_temp_max', 'ush_temp_min', 'ush_temp_media', '_4',
                    'ush_lluvia_mm', 'ush_dias_nieve']

df_meteo = df_meteo[['año', 'mes',
                     'rg_temp_max', 'rg_temp_min', 'rg_temp_media',
                     'rg_lluvia_mm', 'rg_dias_nieve',
                     'ush_temp_max', 'ush_temp_min', 'ush_temp_media',
                     'ush_lluvia_mm', 'ush_dias_nieve']]

# Rellenar año y normalizar nombre de mes
df_meteo['año'] = df_meteo['año'].ffill()
df_meteo['mes'] = df_meteo['mes'].astype(str).str.strip().str.lower()

meses_validos = ['enero','febrero','marzo','abril','mayo','junio',
                 'julio','agosto','septiembre','octubre','noviembre','diciembre']
df_meteo = df_meteo[df_meteo['mes'].isin(meses_validos)]

for col in df_meteo.columns:
    if col != 'mes':
        df_meteo[col] = pd.to_numeric(df_meteo[col], errors='coerce')

df_meteo['año'] = df_meteo['año'].astype(int)
df_meteo = df_meteo[df_meteo['año'] >= 2015]

print("=== METEOROLOGÍA (corregido) ===")
print(f"Filas: {len(df_meteo)} | Columnas: {list(df_meteo.columns)}")
print(f"Período: {df_meteo['año'].min()} – {df_meteo['año'].max()}")
print("\nNulos por columna:")
print(df_meteo.isnull().sum())

print("\nFilas con nulos en Ushuaia (temp_media):")
print(df_meteo[df_meteo['ush_temp_media'].isna()][['año','mes']].head(20))

=== METEOROLOGÍA (corregido) ===
Filas: 132 | Columnas: ['año', 'mes', 'rg_temp_max', 'rg_temp_min', 'rg_temp_media', 'rg_lluvia_mm', 'rg_dias_nieve', 'ush_temp_max', 'ush_temp_min', 'ush_temp_media', 'ush_lluvia_mm', 'ush_dias_nieve']
Período: 2015 – 2025

Nulos por columna:
año                0
mes                0
rg_temp_max        0
rg_temp_min        1
rg_temp_media      0
rg_lluvia_mm       0
rg_dias_nieve     93
ush_temp_max      14
ush_temp_min      12
ush_temp_media    12
ush_lluvia_mm     12
ush_dias_nieve    61
dtype: int64

Filas con nulos en Ushuaia (temp_media):
      año         mes
180  2024       enero
181  2024     febrero
182  2024       marzo
183  2024       abril
184  2024        mayo
185  2024       junio
186  2024       julio
187  2024      agosto
188  2024  septiembre
189  2024     octubre
190  2024   noviembre
191  2024   diciembre


In [4]:
# ── Dataset 3: Pernoctaciones Ushuaia ─────────────────────────────────────
# El archivo tiene 3 hojas de datos separadas por período: 2004-2011, 2012-2023, 2024 en adelante
# Leemos las 3 y las unimos. Solo usamos las columnas de Ushuaia (lado izquierdo del archivo)

meses_lower = ['enero','febrero','marzo','abril','mayo','junio',
               'julio','agosto','septiembre','octubre','noviembre','diciembre']

def cargar_hoja_pernoctaciones(nombre_hoja):
    df = pd.read_excel('16_1_01_Pernoctaciones_ingresos_estadia_promedio.xlsx',
                       sheet_name=nombre_hoja, header=None)
    # Buscar fila de encabezado buscando la palabra "Total"
    fila_inicio = 0
    for i, row in df.iterrows():
        if row.astype(str).str.contains('Total').sum() >= 2:
            fila_inicio = i + 1
            break
    df = pd.read_excel('16_1_01_Pernoctaciones_ingresos_estadia_promedio.xlsx',
                       sheet_name=nombre_hoja, skiprows=fila_inicio, header=None)
    # Tomar columnas de Ushuaia: año, mes, pern_total, viaj_total, estadia_total
    df = df.iloc[:, [0, 1, 2, 7, 12]].copy()
    df.columns = ['año', 'mes', 'ush_pern_total', 'ush_viaj_total', 'ush_estadia_total']
    df['año'] = df['año'].ffill()
    df['mes'] = df['mes'].astype(str).str.strip().str.lower()
    df = df[df['mes'].isin(meses_lower)]
    for col in ['ush_pern_total', 'ush_viaj_total', 'ush_estadia_total']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['año'] = pd.to_numeric(df['año'], errors='coerce').astype('Int64')
    return df

p1 = cargar_hoja_pernoctaciones('por mes 2004-2011')
p2 = cargar_hoja_pernoctaciones('por mes 2012-2023')
p3 = cargar_hoja_pernoctaciones('por mes 2024 en adelante')

df_pern = pd.concat([p1, p2, p3], ignore_index=True)
df_pern = df_pern[df_pern['año'] >= 2015]

print("=== PERNOCTACIONES USHUAIA ===")
print(f"Filas: {len(df_pern)} | Columnas: {list(df_pern.columns)}")
print(f"Período: {df_pern['año'].min()} – {df_pern['año'].max()}")
df_pern.info()
print("\nNulos por columna:")
print(df_pern.isnull().sum())

=== PERNOCTACIONES USHUAIA ===
Filas: 129 | Columnas: ['año', 'mes', 'ush_pern_total', 'ush_viaj_total', 'ush_estadia_total']
Período: 2015 – 2025
<class 'pandas.core.frame.DataFrame'>
Index: 129 entries, 132 to 260
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   año                129 non-null    Int64  
 1   mes                129 non-null    object 
 2   ush_pern_total     127 non-null    float64
 3   ush_viaj_total     127 non-null    float64
 4   ush_estadia_total  127 non-null    float64
dtypes: Int64(1), float64(3), object(1)
memory usage: 6.2+ KB

Nulos por columna:
año                  0
mes                  0
ush_pern_total       2
ush_viaj_total       2
ush_estadia_total    2
dtype: int64


## Decisiones de preprocesamiento

**2020 — excluido en los 3 datasets:**  
El Parque Nacional estuvo cerrado entre abril y agosto de 2020 por la pandemia COVID-19.
Los registros de ese período son cero o nulos, lo que representa una anomalía estructural
y no variación natural del turismo. Incluirlos distorsionaría los patrones estacionales
que el modelo intenta aprender.

**Meteorología 2024 — Río Grande como proxy de Ushuaia:**  
El dataset de temperatura de Ushuaia no registra datos para 2024 en el IPIEC.
Se usarán los datos de Río Grande (misma provincia, clima similar) como variable proxy.
Esta decisión se justifica porque ambas ciudades presentan un comportamiento climático
similar en la serie histórica.

**Período de trabajo final estimado:**  
2015–2025 (excluyendo 2020 y aplicando un lag de 1 mes para transformar el problema
en predicción del mes siguiente).